In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import default_risk.config


application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

application_train_df = pd.merge(application_train_df, 
    bureau_df, 
    on="id_curr", 
    how="left"
)
load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

application_train_df.head()


Columns in application_train_df: ['id_curr', 'days_id_publish']
Columns in bureau_df: ['id_curr', 'id_curr_count', 'id_bureau_prev_1', 'id_bureau_prev_2', 'id_bureau_prev_3']


,id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,credit_type_prev_3,days_credit_update_prev_1,days_credit_update_prev_2,days_credit_update_prev_3,amt_annuity_prev_1,amt_annuity_prev_2,amt_annuity_prev_3,flag_is_present_amt_annuity_prev_1,flag_is_present_amt_annuity_prev_2,flag_is_present_amt_annuity_prev_3
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,Consumer credit,-24.0,-47.0,-34.0,0.0,NaN,0.0,0.0,1.0,0.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,Credit card,-43.0,-550.0,-540.0,NaN,NaN,NaN,1.0,1.0,1.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,None,-382.0,-682.0,NaN,NaN,NaN,NaN,1.0,1.0,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,None,-783.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN


In [2]:
Y= application_train_df["target"]
X= application_train_df.drop(columns=["target"])
X.drop(columns=["id_curr"],inplace=True)

categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype('category')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hiperparams=  {  
    "objective" : 'binary:logistic',
    "random_state" : 42,
    "eval_metric" :"auc",
    "enable_categorical" : True
}

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline+bureau")

[0]	validation_0-auc:0.71408
[1]	validation_0-auc:0.72260
[2]	validation_0-auc:0.72744
[3]	validation_0-auc:0.72960
[4]	validation_0-auc:0.73170
[5]	validation_0-auc:0.73427
[6]	validation_0-auc:0.73735
[7]	validation_0-auc:0.73969
[8]	validation_0-auc:0.74285
[9]	validation_0-auc:0.74424
[10]	validation_0-auc:0.74548
[11]	validation_0-auc:0.74787
[12]	validation_0-auc:0.74857
[13]	validation_0-auc:0.74952
[14]	validation_0-auc:0.75146
[15]	validation_0-auc:0.75238
[16]	validation_0-auc:0.75305
[17]	validation_0-auc:0.75345
[18]	validation_0-auc:0.75397
[19]	validation_0-auc:0.75441
[20]	validation_0-auc:0.75459
[21]	validation_0-auc:0.75521
[22]	validation_0-auc:0.75525
[23]	validation_0-auc:0.75543
[24]	validation_0-auc:0.75561
[25]	validation_0-auc:0.75540
[26]	validation_0-auc:0.75569
[27]	validation_0-auc:0.75624
[28]	validation_0-auc:0.75641
[29]	validation_0-auc:0.75631
[30]	validation_0-auc:0.75622
[31]	validation_0-auc:0.75615
[32]	validation_0-auc:0.75588
[33]	validation_0-au